# PidginMed-ASR — Consistent three-model evaluation

In [1]:
!pip install -q nemo_toolkit[asr] lhotse

In [2]:
!pip install -U jiwer pandas soundfile librosa

In [3]:
from nemo.collections.asr.models import EncDecMultiTaskModel

In [4]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
from pathlib import Path

import ast
import gc
import hashlib
import os
import re

import numpy as np
import pandas as pd
import torch

from jiwer import wer, cer
from tqdm.auto import tqdm


# ============================================================
# BASE PATHS
# ============================================================

BASE_DIR = Path(
    "/content/drive/MyDrive/anon/"
    "pidgindataset2026October/"
)

METADATA_PATH = BASE_DIR / "root" / "Metadata1.csv"
SPLIT_MANIFEST_PATH = BASE_DIR / "shared_split_manifest.csv"

# Fine-tuned models
WHISPER_ROOT = BASE_DIR / "PidginMed-ASR/whisper/hyperparameter_search/trial_01/selected_model/"
CANARY_ROOT = BASE_DIR / "PidginCanary/canary_hyperparameter_tuning/selected_best_model.nemo"
MEDASR_ROOT = BASE_DIR / "medasr_hf_clean/selected_model/"

# Output directory deliberately matches the statistical notebook
RESULTS_DIR = BASE_DIR / "Model_Results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)


print("Metadata:", METADATA_PATH)
print("Split:", SPLIT_MANIFEST_PATH)

print("Whisper:", WHISPER_ROOT)
print("Canary:", CANARY_ROOT)
print("MedASR:", MEDASR_ROOT)

print("Results:", RESULTS_DIR)


DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", DEVICE)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# Check required artifacts before running inference.
for name, p in {"metadata": METADATA_PATH, "split manifest": SPLIT_MANIFEST_PATH,
                "Whisper checkpoint": WHISPER_ROOT, "Canary checkpoint": CANARY_ROOT,
                "MedASR checkpoint": MEDASR_ROOT}.items():
    if not Path(p).exists():
        raise FileNotFoundError(f"Missing {name}: {p}")
    print(f"OK {name}: {p}")

In [7]:
# ============================================================
# NORMALIZATION
# ============================================================

def normalize_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


# ============================================================
# MEDICAL TERM PARSING
# ============================================================

def parse_medical_terms(value):

    if isinstance(value, list):
        terms = value

    elif pd.isna(value):
        terms = []

    elif isinstance(value, str):

        value = value.strip()

        if value in {
            "",
            "[]",
            "[ ]",
            "nan",
            "None",
            "null"
        }:
            terms = []

        else:

            try:
                terms = ast.literal_eval(value)

                if not isinstance(terms, list):
                    terms = [terms]

            except Exception:

                terms = [
                    x.strip()
                    for x in value.split(",")
                    if x.strip()
                ]

    else:
        terms = []


    cleaned = []
    seen = set()

    for term in terms:

        term = normalize_text(term)

        if term and term not in seen:
            cleaned.append(term)
            seen.add(term)

    return cleaned


# ============================================================
# MEDICAL TERM MATCHING
# ============================================================

def phrase_present(text, phrase):

    text = f" {normalize_text(text)} "
    phrase = f" {normalize_text(phrase)} "

    return phrase in text


def reference_medical_terms(reference, terms):

    candidates = set(
        parse_medical_terms(terms)
    )

    return {
        term
        for term in candidates
        if phrase_present(reference, term)
    }


def predicted_medical_terms(prediction, terms):

    candidates = set(
        parse_medical_terms(terms)
    )

    return {
        term
        for term in candidates
        if phrase_present(prediction, term)
    }

In [8]:
# ============================================================
# ORIGINAL CORPUS-LEVEL mWER LOGIC
# ============================================================

def medical_wer(
    predictions,
    references,
    medical_terms_list
):

    ref_med_sentences = []
    pred_med_sentences = []

    for pred, ref, terms in zip(
        predictions,
        references,
        medical_terms_list
    ):

        candidates = set(
            parse_medical_terms(terms)
        )

        ref_present = [
            term
            for term in candidates
            if phrase_present(ref, term)
        ]

        if not ref_present:
            continue

        pred_present = [
            term
            for term in candidates
            if phrase_present(pred, term)
        ]

        ref_med_sentences.append(
            " ".join(sorted(ref_present))
        )

        pred_med_sentences.append(
            " ".join(sorted(pred_present))
        )


    if not ref_med_sentences:
        return np.nan


    return wer(
        ref_med_sentences,
        pred_med_sentences
    )


# ============================================================
# PER-SAMPLE mWER
# ============================================================

def sample_medical_wer(
    prediction,
    reference,
    medical_terms
):

    candidates = set(
        parse_medical_terms(medical_terms)
    )


    ref_present = sorted([
        term
        for term in candidates
        if phrase_present(reference, term)
    ])


    if len(ref_present) == 0:
        return np.nan


    pred_present = sorted([
        term
        for term in candidates
        if phrase_present(prediction, term)
    ])


    reference_medical_string = " ".join(
        ref_present
    )

    prediction_medical_string = " ".join(
        pred_present
    )


    return wer(
        reference_medical_string,
        prediction_medical_string
    )

In [9]:
# ============================================================
# SAMPLE ID -- SAME LOGIC AS YOUR WHISPER NOTEBOOK
# ============================================================

def make_sample_id(row):

    candidate_columns = [
        "sample_id",
        "utterance_id",
        "id",
        "file_id"
    ]

    for col in candidate_columns:

        if (
            col in row.index
            and pd.notna(row[col])
            and str(row[col]).strip()
        ):
            return str(row[col]).strip()


    audio_name = Path(
        str(row["audio_path"])
    ).name

    key = (
        f"{audio_name}|"
        f"{normalize_text(row['pidgin_transcription'])}"
    )

    return hashlib.sha1(
        key.encode("utf-8")
    ).hexdigest()[:20]

In [10]:
def prepare_shared_test_set():

    # --------------------------------------------------------
    # Metadata
    # --------------------------------------------------------

    df = pd.read_csv(
        METADATA_PATH
    )

    df.columns = (
        df.columns
        .str.lower()
        .str.strip()
    )


    # --------------------------------------------------------
    # Support either spelling
    # --------------------------------------------------------

    medical_candidates = [
        "medical_terms_pidgin",
        "medical_terms_pidgen",
        "medical_terms_clean",
        "medical_terms",
    ]

    medical_col = next(
        (
            col
            for col in medical_candidates
            if col in df.columns
        ),
        None
    )

    if medical_col is None:
        raise ValueError(
            "Could not find a medical-term column."
        )


    print(
        "Using medical-term column:",
        medical_col
    )


    # --------------------------------------------------------
    # Standardized medical_terms
    # --------------------------------------------------------

    df["medical_terms"] = (
        df[medical_col]
        .apply(parse_medical_terms)
    )


    # --------------------------------------------------------
    # Reference
    # --------------------------------------------------------

    df["reference"] = (
        df["pidgin_transcription"]
        .apply(normalize_text)
    )


    # --------------------------------------------------------
    # Sample ID
    # --------------------------------------------------------

    if "sample_id" not in df.columns:

        df["sample_id"] = df.apply(
            make_sample_id,
            axis=1
        )

    else:

        missing_id = (
            df["sample_id"].isna()
            |
            (
                df["sample_id"]
                .astype(str)
                .str.strip()
                == ""
            )
        )

        if missing_id.any():

            df.loc[
                missing_id,
                "sample_id"
            ] = df.loc[
                missing_id
            ].apply(
                make_sample_id,
                axis=1
            )


    df["sample_id"] = (
        df["sample_id"]
        .astype(str)
    )


    # --------------------------------------------------------
    # Load shared split
    # --------------------------------------------------------

    manifest = pd.read_csv(
        SPLIT_MANIFEST_PATH,
        dtype={
            "sample_id": str
        }
    )


    if "split" not in manifest.columns:
        raise ValueError(
            "shared_split_manifest.csv "
            "does not contain 'split'."
        )


    manifest["sample_id"] = (
        manifest["sample_id"]
        .astype(str)
    )


    split_lookup = (
        manifest[
            ["sample_id", "split"]
        ]
        .drop_duplicates(
            subset="sample_id"
        )
    )


    # Remove existing split to avoid conflicts
    df = df.drop(
        columns=["split"],
        errors="ignore"
    )


    df = df.merge(
        split_lookup,
        on="sample_id",
        how="inner",
        validate="one_to_one"
    )


    # --------------------------------------------------------
    # TEST ONLY
    # --------------------------------------------------------

    df = df[
        df["split"]
        .astype(str)
        .str.lower()
        .eq("test")
    ].copy()


    # Retain the FULL speaker-held-out test partition here.
    # Medical filtering happens only AFTER all models transcribe it.
    # This permits headline WER/CER on all test utterances.
    full_test_count = len(df)
    if df["sample_id"].duplicated().any():
        raise ValueError("Duplicate sample IDs in test split")
    df["reference_medical_terms"] = df.apply(
        lambda row: sorted(reference_medical_terms(row["reference"], row["medical_terms"])),
        axis=1
    )
    df["medical_eligible"] = df["reference_medical_terms"].apply(lambda x: len(x) > 0)
    print("Full test utterances:", full_test_count)
    print("Eligible medical utterances:", int(df["medical_eligible"].sum()))
    print("Excluded from medical evaluations:", int((~df["medical_eligible"]).sum()))

    # --------------------------------------------------------
    # Resolve audio paths
    # --------------------------------------------------------

    def resolve_audio_path(path_value):

        path = Path(
            str(path_value)
        )

        if path.is_absolute():
            return str(path)

        return str(
            BASE_DIR / path
        )


    df["audio_filepath"] = (
        df["audio_path"]
        .apply(resolve_audio_path)
    )


    # --------------------------------------------------------
    # Verify audio
    # --------------------------------------------------------

    exists = (
        df["audio_filepath"]
        .apply(lambda x: Path(x).exists())
    )


    if not exists.all():

        missing = df.loc[
            ~exists,
            [
                "audio_path",
                "audio_filepath"
            ]
        ]

        print(
            "WARNING: missing audio:",
            len(missing)
        )

        display(
            missing.head(10)
        )

        raise FileNotFoundError(
            "Audio files missing; aborting rather than silently changing the test denominator."
        )


    df = (
        df
        .sort_values("sample_id")
        .reset_index(drop=True)
    )


    print()
    print(
        "FINAL SHARED TEST SAMPLES:",
        len(df)
    )

    print(
        "Unique sample IDs:",
        df["sample_id"].nunique()
    )

    print(
        "Medical-eligible utterances:",
        int(df["medical_eligible"].sum())
    )

    print(
        "Medical exclusions:",
        int((~df["medical_eligible"]).sum())
    )


    return df


test_df = prepare_shared_test_set()

display(
    test_df[
        [
            "sample_id",
            "audio_path",
            "reference",
            "medical_terms"
        ]
    ].head()
)

Using medical-term column: medical_terms_pidgen
Full test utterances: 2245
Eligible medical utterances: 2217
Excluded from medical evaluations: 28

FINAL SHARED TEST SAMPLES: 2245
Unique sample IDs: 2245
Medical-eligible utterances: 2217
Medical exclusions: 28


,sample_id,audio_path,reference,medical_terms
0,0007eceea7445996cb66,root/audio/Speaker_17/Speaker17-129.wav,hello e fit be some age related cortical chang...,"[cortical changes, computed tomography, normal..."
1,001275179d737b980aab,root/audio/Speaker_6/Speaker6-372.wav,after dem pull all my remaining teeth for lowe...,"[lower jaw, lump, cyst, hospital, lump reduce,..."
2,00197304cdccf249be4e,root/audio/Speaker_6/Speaker6-415.wav,my son na five months old and he don dey make ...,"[son, five months old, catarrh phlegm full, ch..."
3,0023b3f0f2acb32f7772,root/audio/Speaker_6/Speaker6-174.wav,go see lung doctor pulmonologist make dem do c...,"[lung, chest x ray spirometry breathing test, ..."
4,002922c6c57a34d7f082,root/audio/Speaker_1/Speaker1-157.wav,a couple of days ago my pulse rate dey fifty e...,"[a couple of days ago, pulse rate dey fifty ei..."


In [11]:
def add_evaluation_metrics(
    frame,
    model_name
):

    frame = frame.copy()


    # --------------------------------------------------------
    # Normalize reference and prediction consistently
    # --------------------------------------------------------

    frame["reference"] = (
        frame["reference"]
        .apply(normalize_text)
    )

    frame["prediction"] = (
        frame["prediction"]
        .apply(normalize_text)
    )


    # --------------------------------------------------------
    # WER
    # --------------------------------------------------------

    frame["wer"] = frame.apply(
        lambda row:
            wer(
                row["reference"],
                row["prediction"]
            ),
        axis=1
    )


    # --------------------------------------------------------
    # CER
    # --------------------------------------------------------

    frame["cer"] = frame.apply(
        lambda row:
            cer(
                row["reference"],
                row["prediction"]
            ),
        axis=1
    )


    # --------------------------------------------------------
    # mWER
    # --------------------------------------------------------

    frame["mwer"] = frame.apply(
        lambda row:
            sample_medical_wer(
                prediction=row["prediction"],
                reference=row["reference"],
                medical_terms=row["medical_terms"],
            ),
        axis=1
    )


    # --------------------------------------------------------
    # Useful downstream medical information
    # --------------------------------------------------------

    frame["reference_medical_terms"] = (
        frame.apply(
            lambda row:
                sorted(
                    reference_medical_terms(
                        row["reference"],
                        row["medical_terms"]
                    )
                ),
            axis=1
        )
    )


    frame["predicted_medical_terms"] = (
        frame.apply(
            lambda row:
                sorted(
                    predicted_medical_terms(
                        row["prediction"],
                        row["medical_terms"]
                    )
                ),
            axis=1
        )
    )


    frame["medical_exact_correct"] = (
        frame.apply(
            lambda row:
                int(
                    set(
                        row["reference_medical_terms"]
                    )
                    ==
                    set(
                        row["predicted_medical_terms"]
                    )
                ),
            axis=1
        )
    )


    frame["medical_num_reference"] = (
        frame[
            "reference_medical_terms"
        ].apply(len)
    )


    frame["medical_num_recovered"] = (
        frame.apply(
            lambda row:
                len(
                    set(
                        row[
                            "reference_medical_terms"
                        ]
                    )
                    &
                    set(
                        row[
                            "predicted_medical_terms"
                        ]
                    )
                ),
            axis=1
        )
    )


    frame["medical_recall"] = (
        frame["medical_num_recovered"] /
        frame["medical_num_reference"].replace(0, np.nan)
    )
    frame["medical_num_predicted"] = frame["predicted_medical_terms"].apply(len)
    frame["medical_tp"] = frame["medical_num_recovered"]
    frame["medical_fp"] = frame["medical_num_predicted"] - frame["medical_tp"]
    frame["medical_fn"] = frame["medical_num_reference"] - frame["medical_tp"]
    # No true reference terms: not a valid medical exact-match trial.
    frame.loc[frame["medical_num_reference"].eq(0), "medical_exact_correct"] = np.nan


    frame["model"] = model_name


    return frame

In [12]:
FEATURE_COLUMNS = [
    "audio_length",
    "speech_rate",
    "signal_to_noise_ratio",
    "silence_ratio",
    "english_ratio",
    "pidgin_ratio",
    "cmi",
    "code_switch_category",
]


def standardized_output(
    evaluated_df
):

    base_columns = [
        "sample_id",
        "audio_path",
        "audio_filepath",
        "speaker_id",
        "pidgin_transcription",
        "reference",
        "prediction",
        "medical_terms",
        "reference_medical_terms",
        "predicted_medical_terms",
        "medical_exact_correct",
        "medical_num_reference",
        "medical_num_recovered",
        "medical_num_predicted",
        "medical_tp",
        "medical_fp",
        "medical_fn",
        "medical_eligible",
        "medical_recall",
        "wer",
        "cer",
        "mwer",
        "model",
    ]


    columns = [
        col
        for col in (
            base_columns
            + FEATURE_COLUMNS
        )
        if col in evaluated_df.columns
    ]


    result = evaluated_df[
        columns
    ].copy()


    # StatisticalSignificanceAnalysis.ipynb expects this
    # terminology in several places.
    result["medical_terms_pidgen"] = (
        result["medical_terms"]
    )


    return result

In [13]:
# ============================================================
# LOAD FINE-TUNED WHISPER-LARGE-V3
# ============================================================

from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
)
import torch


print("Loading Whisper from:")
print(WHISPER_ROOT)


# ------------------------------------------------------------
# Processor
# ------------------------------------------------------------

whisper_processor = WhisperProcessor.from_pretrained(
    str(WHISPER_ROOT)
)


# ------------------------------------------------------------
# Model dtype
# ------------------------------------------------------------

whisper_dtype = (
    torch.float16
    if DEVICE == "cuda"
    else torch.float32
)


# ------------------------------------------------------------
# Model
# ------------------------------------------------------------

whisper_model = WhisperForConditionalGeneration.from_pretrained(
    str(WHISPER_ROOT),
    torch_dtype=whisper_dtype,
)


whisper_model = whisper_model.to(DEVICE)
whisper_model.eval()


# ------------------------------------------------------------
# Generation configuration
# ------------------------------------------------------------

whisper_model.generation_config.language = "en"
whisper_model.generation_config.task = "transcribe"


print("\nWhisper loaded successfully.")
print("Device:", next(whisper_model.parameters()).device)
print("dtype:", next(whisper_model.parameters()).dtype)
print("Model:", whisper_model.config._name_or_path)

Loading Whisper from:
/content/drive/MyDrive/Brock_University/pidgindataset2026October/PidginMed-ASR/whisper/hyperparameter_search/trial_01/selected_model


Loading weights:   0%|          | 0/1259 [00:00<?, ?it/s]


Whisper loaded successfully.
Device: cuda:0
dtype: torch.float16
Model: /content/drive/MyDrive/Brock_University/pidgindataset2026October/PidginMed-ASR/whisper/hyperparameter_search/trial_01/selected_model


In [14]:
# ============================================================
# LOAD FINE-TUNED GOOGLE MEDASR
# ============================================================

from transformers import (
    AutoProcessor,
    AutoModelForCTC,
)


print("Loading MedASR from:")
print(MEDASR_ROOT)


medasr_processor = (
    AutoProcessor.from_pretrained(
        MEDASR_ROOT
    )
)


medasr_model = (
    AutoModelForCTC.from_pretrained(
        MEDASR_ROOT
    )
)


medasr_model.to(DEVICE)
medasr_model.eval()


print()
print("MedASR loaded successfully.")
print("Device:", DEVICE)
print(
    "Parameters:",
    f"{sum(p.numel() for p in medasr_model.parameters()):,}"
)

Loading MedASR from:
/content/drive/MyDrive/Brock_University/pidgindataset2026October/medasr_hf_clean/selected_model


Loading weights:   0%|          | 0/368 [00:00<?, ?it/s]


MedASR loaded successfully.
Device: cuda
Parameters: 105,265,408


In [15]:
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Loading Canary from:")
print(CANARY_ROOT)

canary_model = EncDecMultiTaskModel.restore_from(
    restore_path=str(CANARY_ROOT),
    map_location=torch.device(DEVICE),
)

canary_model = canary_model.to(DEVICE)
canary_model.eval()

print("\n✓ Canary loaded successfully")
print("Device:", next(canary_model.parameters()).device)
print(
    "Parameters:",
    f"{sum(p.numel() for p in canary_model.parameters()):,}"
)

Loading Canary from:
/content/drive/MyDrive/Brock_University/pidgindataset2026October/PidginCanary/canary_hyperparameter_tuning/selected_best_model.nemo
[NeMo I 2026-10-12 19:41:55 mixins:218] _setup_tokenizer: detected an aggregate tokenizer
[NeMo I 2026-10-12 19:41:55 mixins:357] Tokenizer SentencePieceTokenizer initialized with 1152 tokens
[NeMo I 2026-10-12 19:41:55 mixins:357] Tokenizer SentencePieceTokenizer initialized with 1024 tokens
[NeMo I 2026-10-12 19:41:55 mixins:357] Tokenizer SentencePieceTokenizer initialized with 1024 tokens
[NeMo I 2026-10-12 19:41:55 mixins:357] Tokenizer SentencePieceTokenizer initialized with 1024 tokens
[NeMo I 2026-10-12 19:41:55 mixins:357] Tokenizer SentencePieceTokenizer initialized with 1024 tokens
[NeMo I 2026-10-12 19:41:55 aggregate_tokenizer:73] Aggregate vocab size: 5248


[NeMo W 2026-10-12 19:41:55 modelPT:175] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    use_lhotse: true
    tarred_audio_filepaths: null
    manifest_filepath: /content/drive/MyDrive/Brock_University/pidgindataset2026October/PidginCanary/canary_hyperparameter_tuning/manifests/train_canary.jsonl
    sample_rate: 16000
    shuffle: true
    num_workers: 8
    batch_size: 16
    batch_duration: 360
    quadratic_duration: 15
    use_bucketing: true
    num_buckets: 20
    bucket_buffer_size: 20000
    shuffle_buffer_size: 10000
    text_field: text
    lang_field: target_lang
    
[NeMo W 2026-10-12 19:41:55 modelPT:182] If you intend to do validation, please call the ModelPT.setup_validation_data() or ModelPT.setup_multiple_validation_data() method and provide a valid configuration file to setup the validation data loader(s). 
    Validation c

[NeMo I 2026-10-12 19:41:57 save_restore_connector:287] Model EncDecMultiTaskModel was successfully restored from /content/drive/MyDrive/Brock_University/pidgindataset2026October/PidginCanary/canary_hyperparameter_tuning/selected_best_model.nemo.

✓ Canary loaded successfully
Device: cuda:0
Parameters: 182,643,840


In [16]:
# ============================================================
# OUTPUT DIRECTORY
# ============================================================

OUTPUT_DIR = BASE_DIR / "outputRecent"
OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Output directory:")
print(OUTPUT_DIR)


# ============================================================
# COMMON AUDIO LOADER
# ============================================================

import librosa
import soundfile as sf


def load_audio_16k(audio_path):

    audio, sr = sf.read(
        str(audio_path),
        dtype="float32"
    )

    # Stereo / multi-channel -> mono
    if audio.ndim > 1:
        audio = audio.mean(axis=1)

    # Resample if needed
    if sr != 16000:

        audio = librosa.resample(
            audio,
            orig_sr=sr,
            target_sr=16000
        )

        sr = 16000

    return audio.astype(np.float32), sr

Output directory:
/content/drive/MyDrive/Brock_University/pidgindataset2026October/outputRecent


In [19]:

# ============================================================
# EVALUATE FINE-TUNED WHISPER-LARGE-V3 (BATCHED)
# ============================================================

BATCH_SIZE = 32  # Try 8, 16, or 32 depending on GPU memory

whisper_model.eval()
whisper_predictions = []

for start in tqdm(
    range(0, len(test_df), BATCH_SIZE),
    desc="Whisper-large-v3 Batched Inference"
):
    batch_df = test_df.iloc[start:start + BATCH_SIZE]

    # Load audio recordings
    audio_batch = []

    for audio_path in batch_df["audio_filepath"]:
        audio, sr = load_audio_16k(audio_path)
        audio_batch.append(audio)

    # Process and pad the audio batch
    inputs = whisper_processor(
        audio_batch,
        sampling_rate=16000,
        return_tensors="pt",
        padding=True
    )

    input_features = inputs.input_features.to(
        device=DEVICE,
        dtype=whisper_model.dtype
    )

    # Batched inference
    with torch.inference_mode():
        generated_ids = whisper_model.generate(
            input_features,
            language="en",
            task="transcribe",
            max_new_tokens=256
        )

    # Decode the entire batch
    predictions = whisper_processor.batch_decode(
        generated_ids,
        skip_special_tokens=True
    )

    whisper_predictions.extend(
        [normalize_text(p) for p in predictions]
    )

# ------------------------------------------------------------
# Attach predictions
# ------------------------------------------------------------

assert len(whisper_predictions) == len(test_df)

whisper_eval_df = test_df.copy()
whisper_eval_df["prediction"] = whisper_predictions

# ------------------------------------------------------------
# WER / CER / mWER / medical metrics
# ------------------------------------------------------------

whisper_eval_df = add_evaluation_metrics(
    whisper_eval_df,
    model_name="Whisper-large-v3"
)

# ------------------------------------------------------------
# Standardized output
# ------------------------------------------------------------

whisper_output_df = standardized_output(
    whisper_eval_df
)

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

WHISPER_OUTPUT_PATH = (
    OUTPUT_DIR / "Whisper_test_predictions.csv"
)

whisper_output_df.to_csv(
    WHISPER_OUTPUT_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("\nSaved:", WHISPER_OUTPUT_PATH)
print("Rows:", len(whisper_output_df))

display(
    whisper_output_df[
        [
            "sample_id",
            "reference",
            "prediction",
            "wer",
            "cer",
            "mwer",
            "medical_recall"
        ]
    ].head()
)


Whisper-large-v3 Batched Inference:   0%|          | 0/71 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=256) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface


Saved: /content/drive/MyDrive/Brock_University/pidgindataset2026October/outputRecent/Whisper_test_predictions.csv
Rows: 2245


,sample_id,reference,prediction,wer,cer,mwer,medical_recall
0,0007eceea7445996cb66,hello e fit be some age related cortical chang...,how far e fit be some age related critical cha...,0.130435,0.058577,0.500000,0.500000
1,001275179d737b980aab,after dem pull all my remaining teeth for lowe...,after dem pull all my remaining teeth for lowe...,0.043478,0.020528,0.000000,1.000000
2,00197304cdccf249be4e,my son na five months old and he don dey make ...,my son na five months old and he don dey make ...,0.225806,0.151376,0.333333,0.727273
3,0023b3f0f2acb32f7772,go see lung doctor pulmonologist make dem do c...,go see lung doctor make dem do chest x ray spi...,0.107143,0.189542,0.461538,0.750000
4,002922c6c57a34d7f082,a couple of days ago my pulse rate dey fifty e...,a couple of days ago my pulse rate dey now wey...,0.140351,0.120996,0.363636,0.700000


In [20]:
# Full-set WER/CER; medical metrics exclusively on reference-positive rows.
full = whisper_output_df
medical = full.loc[full["medical_num_reference"].gt(0)].copy()
tp = int(medical["medical_tp"].sum())
fp = int(medical["medical_fp"].sum())
fn = int(medical["medical_fn"].sum())
p = tp / (tp + fp) if tp + fp else 0.0
r = tp / (tp + fn) if tp + fn else 0.0
f1 = 2 * p * r / (p + r) if p + r else 0.0
whisper_overall = {
    "model": "Whisper-large-v3",
    "n_test_full": len(full),
    "n_test_medical": len(medical),
    "wer": wer(full["reference"].tolist(), full["prediction"].tolist()),
    "cer": cer(full["reference"].tolist(), full["prediction"].tolist()),
    "mwer": medical_wer(medical["prediction"].tolist(), medical["reference"].tolist(), medical["medical_terms"].tolist()),
    "medical_precision": p,
    "medical_recall": r,
    "medical_f1": f1,
    "medical_exact_match": float(medical["medical_exact_correct"].mean()),
    "medical_tp": tp,
    "medical_fp": fp,
    "medical_fn": fn,
}
print(whisper_overall)

{'model': 'Whisper-large-v3', 'n_test_full': 2245, 'n_test_medical': 2217, 'wer': 0.22321763056929894, 'cer': 0.17675416576120215, 'mwer': 0.26979953992770295, 'medical_precision': 0.9954474214493619, 'medical_recall': 0.7744295418916565, 'medical_f1': 0.8711383972307492, 'medical_exact_match': 0.33062697338746055, 'medical_tp': 13338, 'medical_fp': 61, 'medical_fn': 3885}


In [21]:
# ============================================================
# EVALUATE FINE-TUNED GOOGLE MEDASR
# ============================================================

MEDASR_BATCH_SIZE = 16

medasr_predictions = []


for start in tqdm(
    range(
        0,
        len(test_df),
        MEDASR_BATCH_SIZE
    ),
    desc="Google MedASR"
):

    batch_df = test_df.iloc[
        start:
        start + MEDASR_BATCH_SIZE
    ]


    # --------------------------------------------------------
    # Load audio batch
    # --------------------------------------------------------

    audio_batch = []

    for audio_path in batch_df["audio_filepath"]:

        audio, sr = load_audio_16k(
            audio_path
        )

        audio_batch.append(
            audio
        )


    # --------------------------------------------------------
    # Processor
    # --------------------------------------------------------

    inputs = medasr_processor(
        audio_batch,
        sampling_rate=16000,
        return_tensors="pt",
        padding=True
    )


    # --------------------------------------------------------
    # Determine correct model input automatically
    # --------------------------------------------------------

    model_inputs = {}

    if "input_features" in inputs:

        model_inputs["input_features"] = (
            inputs["input_features"]
            .to(DEVICE)
        )

    elif "input_values" in inputs:

        model_inputs["input_values"] = (
            inputs["input_values"]
            .to(DEVICE)
        )

    else:

        raise KeyError(
            "Processor returned neither "
            "'input_features' nor 'input_values'. "
            f"Available keys: {list(inputs.keys())}"
        )


    # --------------------------------------------------------
    # Optional attention mask
    # --------------------------------------------------------

    if (
        "attention_mask" in inputs
        and
        inputs["attention_mask"] is not None
    ):

        model_inputs["attention_mask"] = (
            inputs["attention_mask"]
            .to(DEVICE)
        )


    # --------------------------------------------------------
    # Inference
    # --------------------------------------------------------

    with torch.inference_mode():

        outputs = medasr_model(
            **model_inputs
        )

        logits = outputs.logits


    # --------------------------------------------------------
    # CTC greedy decoding
    # --------------------------------------------------------

    predicted_ids = torch.argmax(
        logits,
        dim=-1
    )


    batch_predictions = (
        medasr_processor.batch_decode(
            predicted_ids
        )
    )


    medasr_predictions.extend([
        normalize_text(x)
        for x in batch_predictions
    ])


# ============================================================
# VALIDATION
# ============================================================

assert (
    len(medasr_predictions)
    ==
    len(test_df)
), (
    f"Prediction count mismatch: "
    f"{len(medasr_predictions)} "
    f"vs {len(test_df)}"
)


print(
    "Predictions generated:",
    len(medasr_predictions)
)


# ============================================================
# BUILD EVALUATION DATAFRAME
# ============================================================

medasr_eval_df = test_df.copy()

medasr_eval_df[
    "prediction"
] = medasr_predictions


# ============================================================
# WER / CER / mWER / MEDICAL METRICS
# ============================================================

medasr_eval_df = add_evaluation_metrics(
    medasr_eval_df,
    model_name="Google-MedASR"
)


# ============================================================
# STANDARDIZE OUTPUT
# ============================================================

medasr_output_df = standardized_output(
    medasr_eval_df
)


# ============================================================
# SAVE
# ============================================================

MEDASR_OUTPUT_PATH = (
    OUTPUT_DIR
    / "MedASR_test_predictions.csv"
)


medasr_output_df.to_csv(
    MEDASR_OUTPUT_PATH,
    index=False,
    encoding="utf-8-sig"
)


print("\nSaved:")
print(MEDASR_OUTPUT_PATH)

print(
    "Rows:",
    len(medasr_output_df)
)


display(
    medasr_output_df[
        [
            "sample_id",
            "reference",
            "prediction",
            "wer",
            "cer",
            "mwer",
            "medical_recall"
        ]
    ].head()
)

Google MedASR:   0%|          | 0/141 [00:00<?, ?it/s]

Predictions generated: 2245

Saved:
/content/drive/MyDrive/Brock_University/pidgindataset2026October/outputRecent/MedASR_test_predictions.csv
Rows: 2245


,sample_id,reference,prediction,wer,cer,mwer,medical_recall
0,0007eceea7445996cb66,hello e fit be some age related cortical chang...,howver e fit be some age related cortical chan...,0.282609,0.108787,0.500000,0.500000
1,001275179d737b980aab,after dem pull all my remaining teeth for lowe...,after de impve all my remaining teeth for lowe...,0.362319,0.187683,0.500000,0.666667
2,00197304cdccf249be4e,my son na five months old and he don dey make ...,my son na five months old and he don dey make ...,0.268817,0.160550,0.200000,0.909091
3,0023b3f0f2acb32f7772,go see lung doctor pulmonologist make dem do c...,go see lung doctor make dem do check x x ray s...,0.214286,0.228758,0.461538,0.750000
4,002922c6c57a34d7f082,a couple of days ago my pulse rate dey fifty e...,a couple of days ago my pulse rate day fift ei...,0.298246,0.128114,0.409091,0.600000


In [22]:
# Full-set WER/CER; medical metrics exclusively on reference-positive rows.
full = medasr_output_df
medical = full.loc[full["medical_num_reference"].gt(0)].copy()
tp = int(medical["medical_tp"].sum())
fp = int(medical["medical_fp"].sum())
fn = int(medical["medical_fn"].sum())
p = tp / (tp + fp) if tp + fp else 0.0
r = tp / (tp + fn) if tp + fn else 0.0
f1 = 2 * p * r / (p + r) if p + r else 0.0
medasr_overall = {
    "model": "Google-MedASR",
    "n_test_full": len(full),
    "n_test_medical": len(medical),
    "wer": wer(full["reference"].tolist(), full["prediction"].tolist()),
    "cer": cer(full["reference"].tolist(), full["prediction"].tolist()),
    "mwer": medical_wer(medical["prediction"].tolist(), medical["reference"].tolist(), medical["medical_terms"].tolist()),
    "medical_precision": p,
    "medical_recall": r,
    "medical_f1": f1,
    "medical_exact_match": float(medical["medical_exact_correct"].mean()),
    "medical_tp": tp,
    "medical_fp": fp,
    "medical_fn": fn,
}
print(medasr_overall)

{'model': 'Google-MedASR', 'n_test_full': 2245, 'n_test_medical': 2217, 'wer': 0.30267662218547753, 'cer': 0.14830134858010677, 'mwer': 0.41758133420966154, 'medical_precision': 0.9906865353911655, 'medical_recall': 0.6484932938512454, 'medical_f1': 0.7838719865248974, 'medical_exact_match': 0.11772665764546685, 'medical_tp': 11169, 'medical_fp': 105, 'medical_fn': 6054}


In [23]:
# ============================================================
# CANARY OUTPUT NORMALIZER
# ============================================================

def extract_canary_text(output):

    if isinstance(
        output,
        str
    ):
        return output


    if hasattr(
        output,
        "text"
    ):
        return output.text


    if isinstance(
        output,
        dict
    ):

        if "text" in output:
            return output["text"]


    return str(output)

In [24]:
# ============================================================
# EVALUATE FINE-TUNED CANARY-180M
# ============================================================

CANARY_BATCH_SIZE = 16


audio_paths = (
    test_df[
        "audio_filepath"
    ]
    .astype(str)
    .tolist()
)


canary_predictions = []


for start in tqdm(
    range(
        0,
        len(audio_paths),
        CANARY_BATCH_SIZE
    ),
    desc="Canary-180M"
):

    batch_paths = audio_paths[
        start:
        start + CANARY_BATCH_SIZE
    ]


    with torch.inference_mode():

        outputs = canary_model.transcribe(
            batch_paths,
            batch_size=len(
                batch_paths
            )
        )


    for output in outputs:

        prediction = extract_canary_text(
            output
        )

        canary_predictions.append(
            normalize_text(
                prediction
            )
        )


assert (
    len(canary_predictions)
    ==
    len(test_df)
), (
    f"Canary prediction count mismatch: "
    f"{len(canary_predictions)} "
    f"vs {len(test_df)}"
)


# ------------------------------------------------------------
# Evaluation DataFrame
# ------------------------------------------------------------

canary_eval_df = test_df.copy()

canary_eval_df[
    "prediction"
] = canary_predictions


canary_eval_df = add_evaluation_metrics(
    canary_eval_df,
    model_name="Canary-180M"
)


canary_output_df = standardized_output(
    canary_eval_df
)


# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

CANARY_OUTPUT_PATH = (
    OUTPUT_DIR
    / "Canary_test_predictions.csv"
)


canary_output_df.to_csv(
    CANARY_OUTPUT_PATH,
    index=False,
    encoding="utf-8-sig"
)


print("\nSaved:")
print(CANARY_OUTPUT_PATH)

print(
    "Rows:",
    len(canary_output_df)
)

Canary-180M:   0%|          | 0/141 [00:00<?, ?it/s]

[NeMo W 2026-10-12 20:29:34 aed_multitask_models:607] Chunking is disabled. Please pass a single audio file or set batch_size to 1
[NeMo W 2026-10-12 20:29:34 dataloader:881] The following configuration keys are ignored by Lhotse dataloader: enable_chunking,trim_silence
[NeMo W 2026-10-12 20:29:34 dataloader:533] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is noticable, set pretokenize=False in dataloader config.(note: that will disable token-per-second filtering and 2D bucketing features)

Transcribing: 0it [00:00, ?it/s]
Transcribing: 1it [00:04,  4.67s/it]
[NeMo W 2026-10-12 20:29:39 aed_multitask_models:607] Chunking is disabled. Please pass a single audio file or set batch_size to 1
[NeMo W 2026-10-12 20:29:39 dataloader:881] The following configuration keys are ignored by


Saved:
/content/drive/MyDrive/Brock_University/pidgindataset2026October/outputRecent/Canary_test_predictions.csv
Rows: 2245


In [25]:
# Full-set WER/CER; medical metrics exclusively on reference-positive rows.
full = canary_output_df
medical = full.loc[full["medical_num_reference"].gt(0)].copy()
tp = int(medical["medical_tp"].sum())
fp = int(medical["medical_fp"].sum())
fn = int(medical["medical_fn"].sum())
p = tp / (tp + fp) if tp + fp else 0.0
r = tp / (tp + fn) if tp + fn else 0.0
f1 = 2 * p * r / (p + r) if p + r else 0.0
canary_overall = {
    "model": "Canary-180M",
    "n_test_full": len(full),
    "n_test_medical": len(medical),
    "wer": wer(full["reference"].tolist(), full["prediction"].tolist()),
    "cer": cer(full["reference"].tolist(), full["prediction"].tolist()),
    "mwer": medical_wer(medical["prediction"].tolist(), medical["reference"].tolist(), medical["medical_terms"].tolist()),
    "medical_precision": p,
    "medical_recall": r,
    "medical_f1": f1,
    "medical_exact_match": float(medical["medical_exact_correct"].mean()),
    "medical_tp": tp,
    "medical_fp": fp,
    "medical_fn": fn,
}
print(canary_overall)

{'model': 'Canary-180M', 'n_test_full': 2245, 'n_test_medical': 2217, 'wer': 0.2677784810944181, 'cer': 0.20884882364548754, 'mwer': 0.40453499835688467, 'medical_precision': 0.994554057673422, 'medical_recall': 0.6468094989258549, 'medical_f1': 0.7838446383338022, 'medical_exact_match': 0.15877311682453765, 'medical_tp': 11140, 'medical_fp': 61, 'medical_fn': 6083}


In [26]:
# ============================================================
# VERIFY MODEL ALIGNMENT
# ============================================================

assert (
    len(whisper_output_df)
    ==
    len(canary_output_df)
    ==
    len(medasr_output_df)
), "The three models have different numbers of samples."


assert (
    whisper_output_df[
        "sample_id"
    ].tolist()
    ==
    canary_output_df[
        "sample_id"
    ].tolist()
    ==
    medasr_output_df[
        "sample_id"
    ].tolist()
), "sample_id ordering differs between models."


assert (
    whisper_output_df[
        "reference"
    ].tolist()
    ==
    canary_output_df[
        "reference"
    ].tolist()
    ==
    medasr_output_df[
        "reference"
    ].tolist()
), "References differ between model outputs."


assert len(whisper_output_df) == len(test_df)
assert all(x["medical_num_reference"].gt(0).sum() == test_df["medical_eligible"].sum()
           for x in [whisper_output_df, canary_output_df, medasr_output_df])
print("✓ All models evaluated on the same full test set:", len(test_df))
print("✓ Same medical-eligible population:", int(test_df["medical_eligible"].sum()))
print("✓ Sample IDs and references agree")

✓ All models evaluated on the same full test set: 2245
✓ Same medical-eligible population: 2217
✓ Sample IDs and references agree


In [27]:
# ============================================================
# OVERALL RESULTS
# ============================================================

overall_results_df = pd.DataFrame([
    whisper_overall,
    canary_overall,
    medasr_overall
])


OVERALL_RESULTS_PATH = (
    OUTPUT_DIR
    / "overall_model_metrics.csv"
)


overall_results_df.to_csv(
    OVERALL_RESULTS_PATH,
    index=False,
    encoding="utf-8-sig"
)


display(
    overall_results_df
)


print("\nSaved:")
print(OVERALL_RESULTS_PATH)

,model,n_test_full,n_test_medical,wer,cer,mwer,medical_precision,medical_recall,medical_f1,medical_exact_match,medical_tp,medical_fp,medical_fn
0,Whisper-large-v3,2245,2217,0.223218,0.176754,0.269800,0.995447,0.774430,0.871138,0.330627,13338,61,3885
1,Canary-180M,2245,2217,0.267778,0.208849,0.404535,0.994554,0.646809,0.783845,0.158773,11140,61,6083
2,Google-MedASR,2245,2217,0.302677,0.148301,0.417581,0.990687,0.648493,0.783872,0.117727,11169,105,6054



Saved:
/content/drive/MyDrive/Brock_University/pidgindataset2026October/outputRecent/overall_model_metrics.csv


In [29]:
whisper_stats.columns

Index(['sample_id', 'audio_path', 'reference', 'medical_terms',
       'whisper_prediction', 'whisper_wer', 'whisper_cer', 'whisper_mwer',
       'whisper_medical_recall', 'whisper_medical_exact'],
      dtype='str')

In [30]:

# ============================================================
# BUILD ALIGNED STATISTICAL ANALYSIS FILE
# ============================================================

import pandas as pd

# ------------------------------------------------------------
# 1. Common columns and metric mapping
# ------------------------------------------------------------

base_columns = [
    "sample_id",
    "audio_path",
    "reference",
    "medical_terms"
]

metric_columns = [
    "prediction",
    "wer",
    "cer",
    "mwer",
    "medical_recall",
    "medical_exact_correct"
]


# ------------------------------------------------------------
# 2. Prepare standardized model DataFrames
# ------------------------------------------------------------

def prepare_model_stats(
    df,
    model_prefix,
    include_base=False
):
    required_columns = (
        base_columns + metric_columns
        if include_base
        else ["sample_id"] + metric_columns
    )

    missing_columns = [
        col for col in required_columns
        if col not in df.columns
    ]

    if missing_columns:
        raise KeyError(
            f"{model_prefix}: Missing columns: "
            f"{missing_columns}"
        )

    stats = df[required_columns].copy()

    rename_mapping = {
        "prediction": f"{model_prefix}_prediction",
        "wer": f"{model_prefix}_wer",
        "cer": f"{model_prefix}_cer",
        "mwer": f"{model_prefix}_mwer",
        "medical_recall": f"{model_prefix}_medical_recall",
        "medical_exact_correct": f"{model_prefix}_medical_exact"
    }

    stats = stats.rename(columns=rename_mapping)

    # Validate uniqueness before merging
    if stats["sample_id"].duplicated().any():
        raise ValueError(
            f"{model_prefix}: Duplicate sample_id values found."
        )

    return stats


# ------------------------------------------------------------
# 3. Whisper statistics
# ------------------------------------------------------------

whisper_stats = prepare_model_stats(
    whisper_output_df,
    model_prefix="whisper",
    include_base=True
)


# ------------------------------------------------------------
# 4. Canary statistics
# ------------------------------------------------------------

canary_stats = prepare_model_stats(
    canary_output_df,
    model_prefix="canary"
)


# ------------------------------------------------------------
# 5. MedASR statistics
# ------------------------------------------------------------

medasr_stats = prepare_model_stats(
    medasr_output_df,
    model_prefix="medasr"
)


# ------------------------------------------------------------
# 6. Filter medically eligible samples
# ------------------------------------------------------------

whisper_stats = whisper_stats.loc[
    whisper_stats["whisper_medical_exact"].notna()
].copy()

canary_stats = canary_stats.loc[
    canary_stats["canary_medical_exact"].notna()
].copy()

medasr_stats = medasr_stats.loc[
    medasr_stats["medasr_medical_exact"].notna()
].copy()


# ------------------------------------------------------------
# 7. Merge model predictions by sample_id
# ------------------------------------------------------------

aligned_stats_df = (
    whisper_stats
    .merge(
        canary_stats,
        on="sample_id",
        how="inner",
        validate="one_to_one"
    )
    .merge(
        medasr_stats,
        on="sample_id",
        how="inner",
        validate="one_to_one"
    )
)


# ------------------------------------------------------------
# 8. Validate aligned dataset
# ------------------------------------------------------------

expected_samples = int(
    test_df["medical_eligible"].sum()
)

actual_samples = len(aligned_stats_df)

print("Expected medically eligible samples:", expected_samples)
print("Actual aligned samples:", actual_samples)

assert actual_samples == expected_samples, (
    f"Sample mismatch: expected {expected_samples}, "
    f"but obtained {actual_samples}."
)

assert aligned_stats_df["sample_id"].is_unique, (
    "Duplicate sample IDs detected."
)

medical_exact_columns = [
    "whisper_medical_exact",
    "canary_medical_exact",
    "medasr_medical_exact"
]

assert aligned_stats_df[
    medical_exact_columns
].notna().all().all(), (
    "Missing medical exact-match values detected."
)

# Ensure correctness indicators are binary
for col in medical_exact_columns:
    values = aligned_stats_df[col].astype(int)

    assert values.isin([0, 1]).all(), (
        f"Invalid correctness values in {col}."
    )

    aligned_stats_df[col] = values


# ------------------------------------------------------------
# 9. Reorder final columns
# ------------------------------------------------------------

final_columns = [
    "sample_id",
    "audio_path",
    "reference",
    "medical_terms",

    "whisper_prediction",
    "whisper_wer",
    "whisper_cer",
    "whisper_mwer",
    "whisper_medical_recall",
    "whisper_medical_exact",

    "canary_prediction",
    "canary_wer",
    "canary_cer",
    "canary_mwer",
    "canary_medical_recall",
    "canary_medical_exact",

    "medasr_prediction",
    "medasr_wer",
    "medasr_cer",
    "medasr_mwer",
    "medasr_medical_recall",
    "medasr_medical_exact"
]

aligned_stats_df = aligned_stats_df[
    final_columns
].copy()


# ------------------------------------------------------------
# 10. Verify medical exact-match accuracy
# ------------------------------------------------------------

accuracy_results = []

for model in ["whisper", "canary", "medasr"]:

    col = f"{model}_medical_exact"

    correct = int(aligned_stats_df[col].sum())

    accuracy = float(aligned_stats_df[col].mean())

    accuracy_results.append({
        "model": model,
        "correct_utterances": correct,
        "total_utterances": actual_samples,
        "medical_exact_accuracy": accuracy
    })

accuracy_df = pd.DataFrame(accuracy_results)

print("\nMedical Exact-Match Accuracy:")
display(accuracy_df)


# ------------------------------------------------------------
# 11. Save aligned statistical analysis file
# ------------------------------------------------------------

ALIGNED_STATS_PATH = (
    OUTPUT_DIR
    / "aligned_model_predictions_for_statistics.csv"
)

aligned_stats_df.to_csv(
    ALIGNED_STATS_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("\nSaved:", ALIGNED_STATS_PATH)
print("Aligned statistical samples:", len(aligned_stats_df))


# ------------------------------------------------------------
# 12. Preview aligned predictions
# ------------------------------------------------------------

display(aligned_stats_df.head())


Expected medically eligible samples: 2217
Actual aligned samples: 2217

Medical Exact-Match Accuracy:


,model,correct_utterances,total_utterances,medical_exact_accuracy
0,whisper,733,2217,0.330627
1,canary,352,2217,0.158773
2,medasr,261,2217,0.117727



Saved: /content/drive/MyDrive/Brock_University/pidgindataset2026October/outputRecent/aligned_model_predictions_for_statistics.csv
Aligned statistical samples: 2217


,sample_id,audio_path,reference,medical_terms,whisper_prediction,whisper_wer,whisper_cer,whisper_mwer,whisper_medical_recall,whisper_medical_exact,...,canary_cer,canary_mwer,canary_medical_recall,canary_medical_exact,medasr_prediction,medasr_wer,medasr_cer,medasr_mwer,medasr_medical_recall,medasr_medical_exact
0,0007eceea7445996cb66,root/audio/Speaker_17/Speaker17-129.wav,hello e fit be some age related cortical chang...,"[cortical changes, computed tomography, normal...",how far e fit be some age related critical cha...,0.130435,0.058577,0.500000,0.500000,0,...,0.100418,0.833333,0.250000,0,howver e fit be some age related cortical chan...,0.282609,0.108787,0.500000,0.500000,0
1,001275179d737b980aab,root/audio/Speaker_6/Speaker6-372.wav,after dem pull all my remaining teeth for lowe...,"[lower jaw, lump, cyst, hospital, lump reduce,...",after dem pull all my remaining teeth for lowe...,0.043478,0.020528,0.000000,1.000000,1,...,1.000000,1.000000,0.000000,0,after de impve all my remaining teeth for lowe...,0.362319,0.187683,0.500000,0.666667,0
2,00197304cdccf249be4e,root/audio/Speaker_6/Speaker6-415.wav,my son na five months old and he don dey make ...,"[son, five months old, catarrh phlegm full, ch...",my son na five months old and he don dey make ...,0.225806,0.151376,0.333333,0.727273,0,...,0.098624,0.200000,0.909091,0,my son na five months old and he don dey make ...,0.268817,0.160550,0.200000,0.909091,0
3,0023b3f0f2acb32f7772,root/audio/Speaker_6/Speaker6-174.wav,go see lung doctor pulmonologist make dem do c...,"[lung, chest x ray spirometry breathing test, ...",go see lung doctor make dem do chest x ray spi...,0.107143,0.189542,0.461538,0.750000,0,...,1.000000,1.000000,0.000000,0,go see lung doctor make dem do check x x ray s...,0.214286,0.228758,0.461538,0.750000,0
4,002922c6c57a34d7f082,root/audio/Speaker_1/Speaker1-157.wav,a couple of days ago my pulse rate dey fifty e...,"[a couple of days ago, pulse rate dey fifty ei...",a couple of days ago my pulse rate dey now wey...,0.140351,0.120996,0.363636,0.700000,0,...,0.060498,0.318182,0.700000,0,a couple of days ago my pulse rate day fift ei...,0.298246,0.128114,0.409091,0.600000,0


In [31]:
# ============================================================
# SAVE FINAL FILTERED TEST MANIFEST
# ============================================================

final_test_manifest = test_df[
    [
        "sample_id",
        "audio_path",
        "audio_filepath",
        "reference",
        "medical_terms",
        "split",
        "medical_eligible"
    ]
].copy()


TEST_MANIFEST_OUTPUT = (
    OUTPUT_DIR
    / "full_test_manifest_with_medical_eligibility.csv"
)


final_test_manifest.to_csv(
    TEST_MANIFEST_OUTPUT,
    index=False,
    encoding="utf-8-sig"
)


print(
    "Saved:",
    TEST_MANIFEST_OUTPUT
)

Saved: /content/drive/MyDrive/Brock_University/pidgindataset2026October/outputRecent/full_test_manifest_with_medical_eligibility.csv


In [32]:
# Exact McNemar test using the SAME binary strict medical exact-match fields
# that were used to compute each model's medical exact-match accuracy.
from statsmodels.stats.contingency_tables import mcnemar


def paired_mcnemar(df, name_a, col_a, name_b, col_b):
    if df[[col_a, col_b]].isna().any().any():
        raise ValueError("McNemar requires defined correctness on every aligned row")
    a = df[col_a].astype(int).eq(1)
    b = df[col_b].astype(int).eq(1)
    both = int((a & b).sum())
    only_a = int((a & ~b).sum())
    only_b = int((~a & b).sum())
    neither = int((~a & ~b).sum())
    result = mcnemar([[both, only_a], [only_b, neither]], exact=True)
    return dict(model_a=name_a, model_b=name_b, both_correct=both,
                a_correct_b_wrong=only_a, a_wrong_b_correct=only_b,
                both_wrong=neither, statistic=result.statistic,
                p_value=result.pvalue, n=len(df))

specs = [("Whisper", "whisper_medical_exact", "MedASR", "medasr_medical_exact"),
         ("Whisper", "whisper_medical_exact", "Canary", "canary_medical_exact"),
         ("MedASR", "medasr_medical_exact", "Canary", "canary_medical_exact")]
mcnemar_df = pd.DataFrame([paired_mcnemar(aligned_stats_df, *x) for x in specs])
# Check that all pairwise tables imply IDENTICAL model-wise exact-correct totals.
for label, column in [("Whisper", "whisper_medical_exact"),
                      ("MedASR", "medasr_medical_exact"),
                      ("Canary", "canary_medical_exact")]:
    total = int(aligned_stats_df[column].sum())
    print(f"{label}: exact-correct={total}, accuracy={total / len(aligned_stats_df):.6f}")
mcnemar_df.to_csv(OUTPUT_DIR / "mcnemar_strict_exact_match.csv", index=False)
display(mcnemar_df)

Whisper: exact-correct=733, accuracy=0.330627
MedASR: exact-correct=261, accuracy=0.117727
Canary: exact-correct=352, accuracy=0.158773


,model_a,model_b,both_correct,a_correct_b_wrong,a_wrong_b_correct,both_wrong,statistic,p_value,n
0,Whisper,MedASR,214,519,47,1437,47.0,1.190276e-101,2217
1,Whisper,Canary,278,455,74,1410,74.0,6.622842e-68,2217
2,MedASR,Canary,138,123,214,1742,123.0,8.169014e-07,2217
